In [6]:
from vnstock import Company, Listing
import pandas as pd
import time

def get_ticker_list():
    try:
        df = Listing().all_symbols()
        tickers = df["symbol"].astype(str).str.upper().str.strip().unique().tolist()
    except Exception as e:
        print(f"Lỗi lấy danh sách mã: {e}")
        tickers = []
    return tickers

def get_top_3_company_events(symbols, delay=0.5, batch_delay=2):
    all_data = []
    
    for idx, symbol in enumerate(symbols):
        try:
            company = Company(symbol=symbol, source='VCI')
            data = company.events()
            
            if data is not None:
                if isinstance(data, pd.DataFrame) and not data.empty:
                    # Đảm bảo sắp xếp theo ngày công bố mới nhất và lấy 3 sự kiện đầu tiên
                    if 'public_date' in data.columns:
                        data['public_date'] = pd.to_datetime(data['public_date'], format='%d/%m/%Y', errors='coerce')
                        data = data.sort_values(by='public_date', ascending=False).head(3)
                        # Định dạng lại chuỗi nếu cần
                        data['public_date'] = data['public_date'].dt.strftime('%d/%m/%Y')
                    else:
                        data = data.head(3)
                    
                    for _, item in data.iterrows():
                        record = {
                            'symbol': symbol,
                            'event_title': item.get('event_title'),
                            'public_date': item.get('public_date'),
                            'source_url': item.get('source_url'),
                            'event_list_name': item.get('event_list_name'),
                            'event_list_code': item.get('event_list_code'),
                            'id': item.get('id')
                        }
                        all_data.append(record)
                elif isinstance(data, list) and len(data) > 0:
                    # Nếu là list of dicts, sắp xếp theo ngày và lấy 3 phần tử đầu
                    try:
                        sorted_data = sorted(
                            data, 
                            key=lambda x: pd.to_datetime(x.get('public_date'), format='%d/%m/%Y', errors='coerce') if x.get('public_date') else pd.Timestamp.min, 
                            reverse=True
                        )
                    except Exception:
                        sorted_data = data
                    
                    for item in sorted_data[:3]:
                        record = {
                            'symbol': symbol,
                            'event_title': item.get('event_title'),
                            'public_date': item.get('public_date'),
                            'source_url': item.get('source_url'),
                            'event_list_name': item.get('event_list_name'),
                            'event_list_code': item.get('event_list_code'),
                            'id': item.get('id')
                        }
                        all_data.append(record)
            
            print(f"✓ {symbol} ({idx+1}/{len(symbols)})")
        except Exception as e:
            print(f"✗ {symbol}: {e}")
        
        if idx < len(symbols) - 1:
            time.sleep(delay)
        
        if (idx + 1) % 19 == 0:
            print(f"⏸ Nghỉ {batch_delay}s sau {idx + 1} mã...")
            time.sleep(batch_delay)
    
    if all_data:
        return pd.DataFrame(all_data)
    else:
        return pd.DataFrame()

# Chạy thử nghiệm với 5 mã đầu tiên để kiểm tra
tickers = get_ticker_list()[:5]
df_events = get_top_3_company_events(tickers)
display(df_events)

✓ YTC (1/5)
✓ YEG (2/5)
✓ YBM (3/5)
✓ YBC (4/5)
✓ XPH (5/5)


,symbol,event_title,public_date,source_url,event_list_name,event_list_code,id
0,YTC,"YTC - Đăng ký giao dịch bổ sung 6,468,000 cổ p...",NaN,http://fiinpro.com/News/Detail/11262467?lang=v...,Niêm yết thêm,AIS,41244168
1,YTC,YTC-Đăng ký giao dịch bổ sung 280.000 cổ phiếu,NaN,http://fiinpro.com/News/Detail/567655?lang=vi-VN,Niêm yết thêm,AIS,35980
2,YTC,"YTC - Trả cổ tức Đợt 3, 2017 và đợt 1, 2018 bằ...",NaN,http://fiinpro.com/News/Detail/2081035?lang=vi-VN,Trả cổ tức bằng tiền mặt,DIV,35982
3,YEG,YEG - Niêm yết bổ sung 54.800.581 cổ phiếu,NaN,http://fiinpro.com/News/Detail/11520129?lang=v...,Niêm yết thêm,AIS,54273056
4,YEG,"YEG - Niêm yết bổ sung 5,648,190 cổ phiếu",NaN,http://fiinpro.com/News/Detail/11266906?lang=v...,Niêm yết thêm,AIS,41461428
5,YEG,YEG - Niêm yết bổ sung 3.910.000 cổ phiếu,NaN,http://fiinpro.com/News/Detail/2330177?lang=vi-VN,Niêm yết thêm,AIS,35969
6,YBM,YBM - Niêm yết bổ sung 1.2999.942 cổ phiếu,NaN,http://fiinpro.com/News/Detail/4167113?lang=vi-VN,Niêm yết thêm,AIS,35966
7,YBM,YBM - Niêm yết bổ sung 3.574.765 cổ phiếu,NaN,http://fiinpro.com/News/Detail/11839340?lang=v...,Niêm yết thêm,AIS,68817250
8,YBM,"YBM - Trả cổ tức Cả năm, 2018 bằng tiền 1000 V...",NaN,http://cmsv5.stoxplus.com/medialib/Crawler/201...,Trả cổ tức bằng tiền mặt,DIV,35967
9,YBC,YBC - Đăng ký giao dịch bổ sung 2.800.000 cổ p...,NaN,http://fiinpro.com/News/Detail/4747285?lang=vi-VN,Niêm yết thêm,AIS,35963
